In [1]:
"""
PGP 스타일 하이브리드 암호 실습 프로그램

Phil Zimmermann의 Pretty Good Privacy (1991) 동작 원리를 시뮬레이션합니다.
- 하이브리드 암호: 대칭키(메시지 암호화) + 비대칭키(RSA, 세션 키 암호화)
- PGP_설명자료.md와 함께 사용하세요.
"""

import hashlib
import os
import random
from typing import Tuple

# rsa_demo와 동일한 디렉토리에 있으면 RSA 함수 import, 없으면 내부 구현 사용
try:
    from rsa_demo import (
        rsa_keygen,
        rsa_encrypt,
        rsa_decrypt,
        message_to_blocks,
        blocks_to_message,
        rsa_encrypt_message,
        rsa_decrypt_message,
        generate_prime,
    )
    USE_EXTERNAL_RSA = True
except ImportError:
    USE_EXTERNAL_RSA = False


# ---------------------------------------------------------------------------
# RSA 미포함 시 내부 구현 (최소 버전)
# ---------------------------------------------------------------------------

def _gcd(a: int, b: int) -> int:
    while b:
        a, b = b, a % b
    return a


def _extended_gcd(a: int, b: int) -> Tuple[int, int, int]:
    if a == 0:
        return b, 0, 1
    g, x1, y1 = _extended_gcd(b % a, a)
    x = y1 - (b // a) * x1
    y = x1
    return g, x, y


def _mod_inverse(a: int, m: int) -> int:
    g, x, _ = _extended_gcd(a % m, m)
    if g != 1:
        raise ValueError("역원이 존재하지 않음")
    return (x % m + m) % m


def _is_prime(n: int) -> bool:
    if n < 2:
        return False
    if n == 2:
        return True
    if n % 2 == 0:
        return False
    for d in range(3, int(n ** 0.5) + 1, 2):
        if n % d == 0:
            return False
    return True


def _generate_prime(bits: int = 8) -> int:
    low = 2 ** (bits - 1)
    high = 2 ** bits - 1
    while True:
        p = random.randrange(low, high + 1)
        if p % 2 == 0:
            p += 1
        if _is_prime(p):
            return p


def _rsa_keygen(bits: int = 8) -> Tuple[Tuple[int, int], Tuple[int, int]]:
    p = _generate_prime(bits)
    q = _generate_prime(bits)
    while p == q:
        q = _generate_prime(bits)
    n = p * q
    phi = (p - 1) * (q - 1)
    e = 65537
    if e >= phi or _gcd(e, phi) != 1:
        for e in range(3, phi, 2):
            if _gcd(e, phi) == 1:
                break
    d = _mod_inverse(e, phi)
    return (n, e), (n, d)


def _rsa_encrypt(M: int, public_key: Tuple[int, int]) -> int:
    n, e = public_key
    return pow(M, e, n)


def _rsa_decrypt(C: int, private_key: Tuple[int, int]) -> int:
    n, d = private_key
    return pow(C, d, n)


# ---------------------------------------------------------------------------
# 대칭키 암호 (교육용: XOR 기반 스트림 암호)
# 실무 PGP는 IDEA/AES 등 사용
# ---------------------------------------------------------------------------

def symmetric_encrypt(plaintext: bytes, key: bytes) -> bytes:
    """
    대칭키 암호 (간소화): key를 반복하여 XOR
    교육용으로 개념만 시연. 실무에서는 AES 등 사용.
    """
    key_len = len(key)
    return bytes(p ^ key[i % key_len] for i, p in enumerate(plaintext))


def symmetric_decrypt(ciphertext: bytes, key: bytes) -> bytes:
    """XOR은 대칭이므로 암호화와 동일."""
    return symmetric_encrypt(ciphertext, key)


def generate_session_key(key_bits: int = 128) -> bytes:
    """랜덤 세션 키 생성 (PGP에서 IDEA 128비트 등 사용)."""
    return os.urandom(key_bits // 8)


# ---------------------------------------------------------------------------
# 세션 키를 정수로 변환 (RSA 암호화용)
# ---------------------------------------------------------------------------

def key_bytes_to_int(key_bytes: bytes) -> int:
    """바이트 키를 정수로 변환 (RSA 입력용)."""
    return int.from_bytes(key_bytes, byteorder="big")


def int_to_key_bytes(n: int, byte_length: int) -> bytes:
    """정수를 바이트 키로 복원."""
    return n.to_bytes(byte_length, byteorder="big")


# ---------------------------------------------------------------------------
# PGP 스타일 하이브리드 암호
# ---------------------------------------------------------------------------

def _session_key_byte_length(n: int) -> int:
    """n 미만의 정수를 표현할 수 있는 바이트 수 (실습용)."""
    bits = max(8, n.bit_length() - 1)
    return (bits + 7) // 8


def pgp_encrypt(
    message: str,
    recipient_public_key: Tuple[int, int],
) -> Tuple[bytes, int, int]:
    """
    PGP 스타일 암호화.
    1. 랜덤 세션 키 생성 (1 ~ n-1 범위의 정수, RSA로 암호화 가능하도록)
    2. 메시지를 세션 키로 대칭 암호화
    3. 세션 키를 수신자 공개키로 RSA 암호화

    Returns:
        (encrypted_message_bytes, encrypted_session_key_int, session_key_byte_length)
    """
    n, _ = recipient_public_key
    key_bytes = _session_key_byte_length(n)

    # 세션 키: [1, n-1] 범위의 랜덤 정수 (RSA 입력 조건 충족)
    key_int = random.randrange(1, n)
    session_key = int_to_key_bytes(key_int, key_bytes)
    msg_bytes = message.encode("utf-8")

    # 1. 대칭키로 메시지 암호화
    encrypted_msg = symmetric_encrypt(msg_bytes, session_key)

    # 2. 세션 키를 RSA로 암호화
    if USE_EXTERNAL_RSA:
        encrypted_key = rsa_encrypt(key_int, recipient_public_key)
    else:
        encrypted_key = _rsa_encrypt(key_int, recipient_public_key)

    return encrypted_msg, encrypted_key, key_bytes


def pgp_decrypt(
    encrypted_message: bytes,
    encrypted_session_key: int,
    recipient_private_key: Tuple[int, int],
    session_key_bytes: int,
) -> str:
    """
    PGP 스타일 복호화.
    1. 개인키로 세션 키 복호화
    2. 세션 키로 메시지 복호화
    """
    if USE_EXTERNAL_RSA:
        key_int = rsa_decrypt(encrypted_session_key, recipient_private_key)
    else:
        key_int = _rsa_decrypt(encrypted_session_key, recipient_private_key)

    session_key = int_to_key_bytes(key_int, session_key_bytes)
    decrypted_bytes = symmetric_decrypt(encrypted_message, session_key)
    return decrypted_bytes.decode("utf-8")


# ---------------------------------------------------------------------------
# 전자 서명 (선택 기능)
# ---------------------------------------------------------------------------

def pgp_sign(message: str, signer_private_key: Tuple[int, int]) -> int:
    """
    전자 서명: 해시를 개인키로 암호화 (RSA 서명).
    서명 S = H^d mod n (d는 개인 지수)
    Returns: 서명 값 (정수)
    """
    h = hashlib.sha256(message.encode("utf-8")).digest()
    h_int = int.from_bytes(h[:16], byteorder="big")
    n, d = signer_private_key
    if h_int >= n:
        h_int = h_int % n
        if h_int == 0:
            h_int = 1
    return pow(h_int, d, n)  # S = H^d mod n


def pgp_verify(message: str, signature: int, signer_public_key: Tuple[int, int]) -> bool:
    """서명 검증: H' = S^e mod n, H'와 Hash(M) 비교."""
    n, e = signer_public_key
    h_restored = pow(signature, e, n)
    h_actual = int.from_bytes(
        hashlib.sha256(message.encode("utf-8")).digest()[:16],
        byteorder="big"
    )
    if h_actual >= n:
        h_actual = h_actual % n
    return h_restored == h_actual


# ---------------------------------------------------------------------------
# 시연
# ---------------------------------------------------------------------------

def run_demo_hybrid() -> None:
    """하이브리드 암호 시연."""
    print("[1] PGP 스타일 하이브리드 암호 시연")
    print("-" * 50)

    # 수신자 키 생성 (n이 256 이상이어야 세션 키 정수화 가능)
    if USE_EXTERNAL_RSA:
        pub, priv = rsa_keygen(p=61, q=53)  # n=3233
    else:
        pub, priv = _rsa_keygen(bits=12)  # n이 충분히 크도록

    n, e = pub
    print(f"수신자 공개키 (n, e): n={n}, e={e}")

    message = "Hello, PGP! 안녕하세요."
    print(f"평문: \"{message}\"")
    print()

    # 암호화
    enc_msg, enc_key, key_len = pgp_encrypt(message, pub)
    print(f"1) 세션 키 생성 ({key_len * 8}비트 랜덤)")
    print(f"2) 대칭키로 메시지 암호화 → 암호문 길이: {len(enc_msg)} 바이트")
    print(f"3) 세션 키를 RSA로 암호화 → 암호화된 세션 키: {enc_key}")
    print()

    # 복호화
    decrypted = pgp_decrypt(enc_msg, enc_key, priv, key_len)
    print(f"복호화: \"{decrypted}\"")
    print(f"복원 성공: {message == decrypted}")
    print()


def run_demo_flow() -> None:
    """암호화/복호화 흐름 상세 시연."""
    print("[2] 암호화·복호화 흐름 상세")
    print("-" * 50)

    if USE_EXTERNAL_RSA:
        pub, priv = rsa_keygen(p=61, q=53)
    else:
        pub, priv = _rsa_keygen(bits=12)

    n, _ = pub
    key_len = _session_key_byte_length(n)
    msg = "PGP"
    key_int = random.randrange(1, n)
    session_key = int_to_key_bytes(key_int, key_len)
    msg_bytes = msg.encode("utf-8")

    enc_msg = symmetric_encrypt(msg_bytes, session_key)

    if USE_EXTERNAL_RSA:
        enc_key = rsa_encrypt(key_int, pub)
        dec_key = rsa_decrypt(enc_key, priv)
    else:
        enc_key = _rsa_encrypt(key_int, pub)
        dec_key = _rsa_decrypt(enc_key, priv)

    dec_key_bytes = int_to_key_bytes(dec_key, key_len)
    dec_msg = symmetric_decrypt(enc_msg, dec_key_bytes).decode("utf-8")

    print("송신자:")
    print(f"  - 세션 키 K: {session_key.hex()}")
    print(f"  - C_M = SymEnc(M, K) 길이: {len(enc_msg)} 바이트")
    print(f"  - C_K = RSA_Enc(K, 공개키) = {enc_key}")
    print()
    print("수신자:")
    print(f"  - K' = RSA_Dec(C_K, 개인키) = {dec_key}")
    print(f"  - M' = SymDec(C_M, K') = \"{dec_msg}\"")
    print(f"  - 일치: {msg == dec_msg}")
    print()


def run_demo_signature() -> None:
    """전자 서명 시연."""
    print("[3] 전자 서명 시연")
    print("-" * 50)

    if USE_EXTERNAL_RSA:
        pub, priv = rsa_keygen(p=61, q=53)
    else:
        pub, priv = _rsa_keygen(bits=12)

    message = "서명할 메시지"
    sig = pgp_sign(message, priv)
    ok = pgp_verify(message, sig, pub)

    print(f"메시지: \"{message}\"")
    print(f"서명 (해시를 개인키로 암호화): {sig}")
    print(f"검증 (공개키로 복원 후 비교): {'성공' if ok else '실패'}")

    # 위변조 시 검증 실패
    tampered = "변조된 메시지"
    ok2 = pgp_verify(tampered, sig, pub)
    print(f"위변조 메시지 검증: {'성공' if ok2 else '실패 (예상됨)'}")
    print()


# ---------------------------------------------------------------------------
# 메인
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    print("=" * 60)
    print("PGP 스타일 하이브리드 암호 실습")
    print("(Phil Zimmermann, Pretty Good Privacy, 1991)")
    print("=" * 60)
    print()

    run_demo_hybrid()
    run_demo_flow()
    run_demo_signature()

    print("=" * 60)
    print("실습 완료. PGP_설명자료.md를 참고하세요.")
    print("=" * 60)


PGP 스타일 하이브리드 암호 실습
(Phil Zimmermann, Pretty Good Privacy, 1991)

[1] PGP 스타일 하이브리드 암호 시연
--------------------------------------------------
수신자 공개키 (n, e): n=3233, e=7
평문: "Hello, PGP! 안녕하세요."

1) 세션 키 생성 (16비트 랜덤)
2) 대칭키로 메시지 암호화 → 암호문 길이: 28 바이트
3) 세션 키를 RSA로 암호화 → 암호화된 세션 키: 530

복호화: "Hello, PGP! 안녕하세요."
복원 성공: True

[2] 암호화·복호화 흐름 상세
--------------------------------------------------
송신자:
  - 세션 키 K: 0ba7
  - C_M = SymEnc(M, K) 길이: 3 바이트
  - C_K = RSA_Enc(K, 공개키) = 1579

수신자:
  - K' = RSA_Dec(C_K, 개인키) = 2983
  - M' = SymDec(C_M, K') = "PGP"
  - 일치: True

[3] 전자 서명 시연
--------------------------------------------------
메시지: "서명할 메시지"
서명 (해시를 개인키로 암호화): 2630
검증 (공개키로 복원 후 비교): 성공
위변조 메시지 검증: 실패 (예상됨)

실습 완료. PGP_설명자료.md를 참고하세요.
